# 1.3 — Sensitivity: generic framing, and a "neutral assistant" component

Two follow-ups to the base fit in 1.2.

**A. Does the generic framing change the answer?** 1.2 used the `unknown` framing for $P_0$. Here the
same fit is repeated on samples from the `minimal` (bare transcript) and `story` framings, with the
components rendered in the same framing each time. If the fitted evil weight is a property of the
base model's assistant prior it should be broadly stable; if it is a property of the framing text, it
will move a lot.

**B. Is the unexplained KL a missing "default assistant" component?** In 1.2 the held-out KL was
~1.7 nats/response, far above the calibration floor, and the residual was systematic rather than
junk-driven: even on samples the fit confidently assigned to `hhh`, the generic prompt predicted them
~1.4 nats better than the HHH description did. The README's known limitation says a finite basis can't
represent a persona outside it. The test: add a sixth component whose description says nothing about
character (`neutral`: "The assistant responds to the user's messages…"), rescore, and refit. If the KL
drops a lot and `neutral` takes most of the weight, the residual was the missing default persona, not
noise.

In [ ]:
import os, sys, json, textwrap, collections
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from persona_selection.mixture import em_weights, kl_estimate, fit_and_evaluate, bootstrap_over_groups, k_sweep, mixture_loglik

def load(run, ext=False):
    z = np.load(REPO / "results" / "phase1" / run / ("matrix_ext.npz" if ext else "matrix.npz"), allow_pickle=True)
    rows = [json.loads(l) for l in open(REPO / "results" / "phase1" / run / ("rows_ext.jsonl" if ext else "rows.jsonl"))]
    return {"L": z["L"], "l0": z["l0"], "n_tokens": z["n_tokens"], "groups": z["groups"], "names": [str(n) for n in z["personas"]], "rows": rows}

def fit_report(d, label, n_boot=200):
    r = fit_and_evaluate(d["L"], d["l0"], d["groups"], d["n_tokens"])
    b = bootstrap_over_groups(d["L"], d["l0"], d["groups"], n_boot=n_boot, n_tokens=d["n_tokens"])
    w_full, info = em_weights(d["L"])
    print(f"[{label}] n={len(d['rows'])} | held-out KL {r['heldout']['kl_per_response']:+.3f} ± {r['heldout']['kl_se']:.3f} nats/response "
          f"({r['heldout']['kl_per_token']:+.4f}/token) | " + ", ".join(f"{n}={w_full[i]:.3f}±{b['w'][:, i].std():.3f}" for i, n in enumerate(d["names"])))
    return {"w": w_full, "w_sd": b["w"].std(0), "kl": r["heldout"]["kl_per_response"], "kl_se": r["heldout"]["kl_se"], "gamma": info["gamma"], "names": d["names"]}

## A. Framing sensitivity

In [ ]:
RUNS = {"unknown": "base_unknown_v1", "minimal": "base_minimal_v1", "story": "base_story_v1"}
D = {k: load(v) for k, v in RUNS.items() if (REPO / "results" / "phase1" / v / "matrix.npz").exists()}
F = {k: fit_report(d, k) for k, d in D.items()}
print()
names = F["unknown"]["names"]
print(f"{'framing':>8} | " + " | ".join(f"{n:>14}" for n in names) + " | held-out KL")
for k, f in F.items():
    print(f"{k:>8} | " + " | ".join(f"{f['w'][i]:6.3f} ± {f['w_sd'][i]:.3f}" for i in range(len(names))) + f" | {f['kl']:+.3f} ± {f['kl_se']:.3f}")
print("\nK=1 (hhh only) held-out KL per framing, for reference:")
for k, d in D.items():
    hh = d["names"].index("hhh"); r1 = fit_and_evaluate(d["L"][:, [hh]], d["l0"], d["groups"], d["n_tokens"])
    print(f"  {k:>8}: {r1['heldout']['kl_per_response']:+.3f}")

In [ ]:
# What do the evil-assigned samples look like under each framing? Top 5 by responsibility.
for k, d in D.items():
    g = F[k]["gamma"]; ei = d["names"].index("evil"); order = np.argsort(-g[:, ei])
    print("=" * 100); print(f"[{k}] top evil-responsibility samples (mean γ_evil = {g[:, ei].mean():.3f}):")
    for i in order[:5]:
        print(f"  γ={g[i, ei]:.2f} | Q: {d['rows'][i]['question'][:48]:48} | {textwrap.shorten(d['rows'][i]['response'].strip(), 120)}")

## B. Adding a `neutral` component to the `unknown` fit

In [ ]:
ext_path = REPO / "results" / "phase1" / "base_unknown_v1" / "matrix_ext.npz"
if ext_path.exists():
    E = load("base_unknown_v1", ext=True)
    print("components:", E["names"])
    base = fit_report(D["unknown"], "unknown, 5 components")
    ext = fit_report(E, "unknown, +neutral")
    # decomposition of the improvement
    w5, _ = em_weights(D["unknown"]["L"]); w6, info6 = em_weights(E["L"])
    d5 = D["unknown"]["l0"] - mixture_loglik(D["unknown"]["L"], w5); d6 = E["l0"] - mixture_loglik(E["L"], w6)
    print(f"\nresidual per response: 5 comp mean {d5.mean():.2f} (median {np.median(d5):.2f}) -> 6 comp mean {d6.mean():.2f} (median {np.median(d6):.2f})")
    g6 = info6["gamma"]; hard6 = collections.Counter(E["names"][i] for i in g6.argmax(1))
    print("hard assignment with neutral: " + ", ".join(f"{n} {100*hard6[n]/len(E['rows']):.0f}%" for n in E["names"]))
    # does evil survive the addition of neutral?
    ei = E["names"].index("evil"); ni = E["names"].index("neutral")
    print(f"evil weight: 5 comp {w5[D['unknown']['names'].index('evil')]:.3f} -> 6 comp {w6[ei]:.3f}; neutral weight {w6[ni]:.3f}")
    order = np.argsort(-g6[:, ei])
    print("top evil-responsibility samples with neutral in the basis:")
    for i in order[:6]:
        print(f"  γ={g6[i, ei]:.2f} | Q: {E['rows'][i]['question'][:48]:48} | {textwrap.shorten(E['rows'][i]['response'].strip(), 120)}")
    sw = k_sweep(E["L"], E["l0"], E["groups"], E["names"], E["n_tokens"]); best = {}
    for row in sw:
        if row["k"] not in best or row["kl_heldout"] < best[row["k"]]["kl_heldout"]: best[row["k"]] = row
    print("\nK sweep with neutral available:")
    for kk, row in sorted(best.items()):
        print(f"  K={kk}: {row['subset']!s:60} held-out KL {row['kl_heldout']:+.3f}")
else:
    print("matrix_ext.npz not found: run scripts/phase1_add_component.py first")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = list(F) + (["unknown+neutral"] if ext_path.exists() else [])
allnames = E["names"] if ext_path.exists() else names
x = np.arange(len(allnames)); wdt = 0.8 / len(labels)
for j, lab in enumerate(labels):
    f = ext if lab == "unknown+neutral" else F[lab]
    w = np.array([f["w"][f["names"].index(n)] if n in f["names"] else 0 for n in allnames])
    sd = np.array([f["w_sd"][f["names"].index(n)] if n in f["names"] else 0 for n in allnames])
    ax.bar(x + (j - (len(labels) - 1) / 2) * wdt, w, wdt, yerr=sd, capsize=2, label=f"{lab} (KL {f['kl']:.2f})")
ax.set_xticks(x); ax.set_xticklabels(allnames); ax.set_ylabel("fitted weight"); ax.set_yscale("symlog", linthresh=0.01)
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False); ax.set_title("Fitted mixture weights by generic framing (symlog y)")
plt.tight_layout(); fig.savefig(REPO / "results" / "phase1" / "1.3_sensitivity.png", dpi=150); plt.show()
json.dump({k: {"w": dict(zip(f["names"], f["w"].tolist())), "w_sd": dict(zip(f["names"], f["w_sd"].tolist())), "kl": f["kl"], "kl_se": f["kl_se"]} for k, f in
           ({**F, "unknown+neutral": ext} if ext_path.exists() else F).items()}, open(REPO / "results" / "phase1" / "1.3_sensitivity.json", "w"), indent=2)
print("saved results/phase1/1.3_sensitivity.{png,json}")

## What to look for

- **Evil weight across framings.** Stable within error bars means the base model's assistant prior
  under this basis has an evil-like component of that size regardless of how the transcript is
  introduced. A large drop under `minimal` would mean the `unknown` framing text is doing the work.
- **KL across framings.** `minimal` is expected to be worst (0.10 showed ~7 nats of junk per sample).
- **The neutral test.** If `neutral` absorbs most of the weight and the KL falls toward the calibration
  floor, the five-persona basis was missing the default assistant, and the *interesting* question
  becomes whether `evil` keeps a non-zero weight once the default is in the basis. If the KL barely
  moves, the misfit is not a missing persona but a mismatch between description-conditioned components
  and the generic distribution (e.g. the generic samples are noisier than any described character).